In [1]:
from abc import ABC, abstractmethod
import re
import pandas as pd
import numpy as np
from google.colab import files
import io
from collections.abc import Iterator

### Exercici 1: Selecció de Personal

> Tu treballes com a analista de dades en el departament de RRHH d'una empresa. S'ha obert una nova oferta laboral interna i l'empresa ha decidit que només les primeres 10 persones que s'inscriguin podran participar en el procés de selecció.

Per això, t'han encarregat desenvolupar una eina senzilla en Python que actuï com a formulari d'inscripció i sistema de classificació inicial. Aquest formulari ha de permetre introduir, per a cada candidat/a:

- Nom i Cognoms
- Telèfon de contacte
- Anys d'experiència dins de l'empresa
- Coneixement de Català (0–3 punts)
    - 0 = sense coneixement
    - 1 = fins a nivell B1
    - 2 = nivell B2
    - 3 = C1 o superior
- Domini d'Excel (0–3 punts)
    - 0 = nivell bàsic
    - 1 = ús de funcions
    - 2 = taules dinàmiques
    - 3 = Visual Basic i Macros
- Títol acadèmic (0–3 punts)
    - 0 = ESO
    - 1 = FP
    - 2 = Universitat
    - 3 = Màster o superior

Assegurant-se que tota la informació introduïda és correcta i entra dins els valors permesos. A partir d'aquestes dades, el programa haurà de calcular una puntuació total aplicant el pes de l'experiència (anys * 1.5) i sumant-hi els punts de català, excel i titulació, i a continuació classificar automàticament cada persona segons els llindars establerts per l'empresa:

- Prioritat Alta → puntuació ≥ 12
- Candidat adequat → 8 ≤ puntuació < 12
- No prioritari → puntuació < 8

Cada vegada que algú complet a el formulari, s'ha de mostrar la seva puntuació individual. Quan ja s'hagin inscrit exactament deu persones, l'eina ha de deixar d'acceptar nous registres i generar un informe final en pantalla, ordenat primer segons la prioritat i, en cas d'empat, per la puntuació obtinguda. Aquest informe ha d'incloure el nom, el telèfon i la puntuació de cada candidat per tal que el departament de RRHH els pugui contactar ràpidament.

**Important**

Per resoldre aquest exercici, hauràs de:

- Desenvolupar un sistema modular amb les funcions següents:
    - `obtenir_dades`
    - `validar_dades`
    - `calcular_puntuacio`
    - `classificar_candidat`
    - `mostrar_individual`
    - `mostrar_informe`
    - `main_seleccio` # orquestra totes les funcions
- Escriure primer l'algoritme pas a pas, indicant clarament quines operacions farà cada funció.
- Escriure el codi de cada funció.

In [2]:
inputs_test = [
  ["Laura Serra", "608112233", "6", "3", "3", "3"],
  ["Clara Gómez", "655221199", "3", "3", "3", "2"],
  ["Sergio López", "644112211", "3", "2", "2", "2"],
  ["Candidat Error", "611000000", "5", "7", "3", "2"], # Català = 7 → NO VÀLID
  ["Marta Ruiz", "699887766", "2", "3", "2", "1"],
  ["Marc Vidal", "612443322", "6", "3", "3", "3"],
  ["Júlia Romero", "677998877", "5", "3", "3", "2"],
  ["Pol Navarro", "699554433", "4", "3", "2", "2"],
  ["David Ferrer", "677665544", "2", "1", "1", "1"],
  ["Anna Paredes", "633221144", "1", "2", "1", "0"],
  ["Anibal Paredes", "644272727", "1", "2", "1", "0"],
  ["Núria Casals", "600112200", "8", "2", "3", "3"] # 11è candidat
]

#### Data Ingestion

The following code is in charge of extracting user input using standard I/O and pertains the modules `obtenir_dades` and `validar_dades`.

- `abstract class` `InputField`: When the class is built, it should contain a `field` variable with the cleaned and parsed data extracted from user input.

    - `__init__(self, s: str | None)`: Initializes the class. When `s` is not `None` it tries to validate, clean and parse the string. If the validation fails, it enters a loop demanding user input.
        1. Set `valid` to `False`.
        2. If `s` is not `None`:
            1. Try to validate, clean and parse `s`, `valid` is set to `True` only if `s` passed validation.
        3. If `valid` is `True`, exit execution.
        4. Take user input and put it in `s`.
        5. Try to validate, clean and parse `s`, `valid` is set to `True` pnly if `s` passed validation.
        6. Go to to step `3`.
  
    - `take_input(self) -> str`: Abstract function (subclasses have to define it). This function has to take user input and return it. This function pertains the module `obtenir_dades`.

    - `validate(self, s: str) -> bool`: Abstract function (subclasses have to define it). This function has to validate `s`. If `s` is valid, it returns `True` and stores the parsed and cleaned data in `self.field`, else it returns `False`.

    - `__str__(self)`: This defines how the class is shown through print.


In [3]:

class InputField(ABC):
  def __init__(self, s: str | None = None):
    valid: bool = False
    if s is not None:
      valid = self.validate(s)
    while not valid:
      s = self.take_input()
      valid = self.validate(s)
  @abstractmethod
  def take_input(self) -> str:
    pass
  @abstractmethod
  def validate(self, s: str) -> bool:
    pass
  @abstractmethod
  def __str__(self) -> str:
    pass

- `class` `NameField(InputField)`: When the class is built, it should contain the user's name/surname in `self.field`, obtained through user input, or through a string parameter provided on construction.

    - `take_input(self) -> str`:
        1. Propmt the user to write their name and surname.
        2. Return the inputed string.

    - `validate(self, s: str) -> bool`:
        1. Strip leading and trailing whitespace characters from `s`.
        2. If `s` is less than `2` characters long:
            1. Print reason of rejection.
            2. Return `False`.
        3. Put `s` in `self.field`
        4. Return `True`.

In [4]:
class NameField(InputField):
  def take_input(self) -> str:
    return input("Input name and surname: ")
  def validate(self, s: str) -> bool:
    s = s.strip()
    if len(s) < 2:
      print("Has to be at least 2 characters long.")
      return False
    self.field: str = s
    return True
  def __str__(self) -> str:
    return self.field

- `class` `PhoneField(InputField)`: When the class is built, it should contain the user's phone in `self.field`, obtained through user input, or through a string parameter provided on construction.

    - `take_input(self) -> str`:
        1. Propmt the user to write their phone number.
        2. Return the inputed string.

    - `validate(self, s: str) -> bool`:
        1. Strip leading and trailing whitespace characters from `s`.
        2. Check if `s` matches the regex expression `r"\+?[\d\s\-()]+"`, a state machine traverses the string character by character:
            1. Is the character a `+`?
                - True: the state machine skips one character, goto state `2`.
                - False: goto state `2`.
            2. Is the character either a digit, whitespace, `-`, `(` or `)`?
                - True: the state machine skips one character, goto state `3`.
                - False: the state machine rejects the match.
            3. Has the string terminated?
                - True: the string matches the regex.
                - False: goto state `2`.
        3. If `s` doesn't match the regex:
            1. Print reason of rejection.
            2. Return False
        4. Substitute all the substrings in `s` that match the regex expression `r"\D"` with the string `""` ans store them in `d`, i.e. remove all substrings that match the regex. The regex matches all non-digit characters. The result in `d` are all the digits in `s` in their original order and nothing else.
        5. If `d` is less than `7` characters long:
            1. Print reason of rejection.
            2. Return `False`.
        6. If `d` is more than `15` characters long:
            1. Print reason of rejection.
            2. Return `False`.
        7. If `s` starts with `+`:
            - True: `p` stores the string `"+"`.
            - False: `p` stores an empty string `""`.
        8. Store the concatenation of `p` and `d` in `self.field`.
        4. Return `True`.

In [5]:
class PhoneField(InputField):
  def take_input(self):
    return input("Input phone number: ")
  def validate(self, s: str) -> bool:
    s = s.strip()
    if not re.fullmatch(r"\+?[\d\s\-()]+", s):
      print(
        "Invalid number: a valid phone may begin with a '+', "
        "contain digits, spaces, dashes or parentheses."
      )
      return False
    d = re.sub(r"\D", "", s)
    if len(d) < 7:
      print(
        f"Too short: input has {len(d)} digits, "
        "need at least 7."
      )
      return False
    if len(d) > 15:
      print(
        f"Too long: input has {len(d)} digits, "
        "there can be 15 at most."
      )
      return False
    p = "+" if s.startswith("+") else ""
    self.field: str = p + d
    return True
  def __str__(self) -> str:
    return self.field


- `class` `YearsExperienceField(InputField)`: When the class is built, it should contain the user's years of experience in `self.field`, obtained through user input, or through a string parameter provided on construction.

    - `take_input(self) -> str`:
        1. Propmt the user to write their years of experience.
        2. Return the inputed string.

    - `validate(self, s: str) -> bool`:
        1. Strip leading and trailing whitespace from `s`.
        2. Try to parse the resulting string into a `float` and store it in `yrs`:
            - Fail: Print reason / instructions to get it right next time, and return `False`.
        3. If `yrs` is negative:
            1. Print rejection reason.
            2. Return `False`.
        4. If `yrs` is greater or equal to `80`:
            1. Print rejection reason.
            2. Return `False`.
        5. Store `yrs` value in `self.field`..
        6. Return `True`.

In [6]:
class YearsExperienceField(InputField):
  def take_input(self) -> str:
    return input("Intorduce years of experience: ")
  def validate(self, s: str) -> bool:
    try:
      yrs = float(s.strip())
      if yrs < 0:
        print("Invalid input number: has to be positive.")
        return False
      if yrs >= 80:
        print("Invalid input number: has to be less than 80.")
        return False
      self.field: float = yrs
      return True
    except (TypeError, ValueError):
      print("Failed to parse value: has to be a positive real number.")
    return False
  def __str__(self) -> str:
    return f"{self.field}"

- `class` `LVLField(InputField)`: When the class is built, it should contain the user's input in `self.field`, obtained through user input, or through a string parameter provided on construction. This subclass accepts only the inputs `0`, `1`, `2` or `3`.

    - `validate(self, s: str) -> bool`:
        1. Strip leading and trailing whitespace from `s`.
        2. Try to parse the resulting string into an `int` and store it in `lvl`:
            - Fail: Print reason / instructions to get it right next time, and return `False`.
        3. If `lvl` is either `0`, `1`, `2` or `3`:
            1. Store `lvl` value in `self.field`
            2. Return `True`.
        4. Print Reason of rejection.
        5. Return `False`.

In [7]:
class LVLField(InputField):
  def validate(self, s: str) -> bool:
    try:
      lvl = int(s.strip())
      if 0 <= lvl <= 3:
        self.field: int = lvl
        return True
      print("Input has to be either 0, 1, 2 or 3.")
    except (TypeError, ValueError):
      print(
        "Failed to parse value:"
        "input has to be either 0, 1, 2 or 3."
      )
    return False

- `class` `CatalanLVLField(LVLField)`: When the class is built, it should contain the user's catalan level in `self.field`, obtained through user input, or through a string parameter provided on construction. This subclass accepts only the inputs `0`, `1`, `2` or `3`.

    - `take_input(self) -> str`:
        1. Propmt the user to write their years of experience.
        2. Return the inputed string.

In [8]:
class CatalanLVLField(LVLField):
  def take_input(self) -> str:
    return input(
      "Input catalan lvl:\n"
      "  0 → no knowledge\n"
      "  1 → A1 to B1\n"
      "  2 → B2\n"
      "  3 → C1 or superior\n"
    )
  def __str__(self) -> str:
    match self.field:
      case 0: return "None"
      case 1: return "A1-B1"
      case 2: return "B2"
      case 3: return "C1-C2 or Native"
      case _: return "Error"

- `class` `ExcelLVLField(LVLField)`: When the class is built, it should contain the user's excel level in `self.field`, obtained through user input, or through a string parameter provided on construction. This subclass accepts only the inputs `0`, `1`, `2` or `3`.

    - `take_input(self) -> str`:
        1. Propmt the user to write their years of experience.
        2. Return the inputed string.

In [9]:
class ExcelLVLField(LVLField):
  def take_input(self) -> str:
    return input(
      "Input excel lvl:\n"
      "  0 → basic\n"
      "  1 → use of functions\n"
      "  2 → dinamic tables\n"
      "  3 → visual basic & macros\n"
    )
  def __str__(self) -> str:
    match self.field:
      case 0: return "Basic"
      case 1: return "Knows functions"
      case 2: return "Knows dynamic tables"
      case 3: return "Knows Visual Basic and Macros"
      case _: return "Error"

- `class` `AcademicLVLField(LVLField)`: When the class is built, it should contain the user's academic level in `self.field`, obtained through user input, or through a string parameter provided on construction. This subclass accepts only the inputs `0`, `1`, `2` or `3`.

    - `take_input(self) -> str`:
        1. Propmt the user to write their years of experience.
        2. Return the inputed string.

In [10]:
class AcademicLVLField(LVLField):
  def take_input(self) -> str:
    return input(
      "Input academic lvl:\n"
      "  0 → ESO\n"
      "  1 → FP\n"
      "  2 → University\n"
      "  3 → Master or superior\n"
    )
  def __str__(self) -> str:
    match self.field:
      case 0: return "ESO"
      case 1: return "FP"
      case 2: return "University"
      case 3: return "Masters or superior"
      case _: return "Error"

- `class` `Nonerator`: Placeholder iterator that always returns `None`.

In [11]:
class Nonerator:
  def __init__(self):
    return
  def __next__(self):
    return None

- `class` `Candidate`: This class represents a single candidate.

    - `__init__(self, data: list[str] | None)`: Initializes the class by initializing all the fields, which in turn process `data` if it is not `None` or try to get filled out from user interaction through standard I/O.

    - `priority() -> str`:
        1. If `pts` is greater or equal than `12`:
            - Return `"High Priority"`
        2. If `pts` is greater or equal than `8`:
            - Return `"Adequate"`
        3. Else return `"No priority"`

    - `pts() -> float`: Calculates puntuation.
        1. Multiplies the years of experience by `1.5`.
        2. Converts the catalan level `int` to `float` and sums it.
        3. Converts the excel level `int` to `float` and sums it.
        4. Converts the academic level `int` to `float` and sums it.
        5. Returns the resulting sum.

    - `__str__() -> str`: Defines the string shown on `print()`. Various strings are concatenated with end of line characters `\n` and using the `__str__()` definitions of each `InputField` class. The resulting string includes all the fields and the user's puntuation and priority.

In [12]:
class Candidate:
  def __init__(self, data: list[str] | None = None):
    it = Nonerator() if data is None else iter(data)
    self.name         = NameField(next(it))
    self.phone        = PhoneField(next(it))
    self.years_exp    = YearsExperienceField(next(it))
    self.catalan_lvl  = CatalanLVLField(next(it))
    self.excel_lvl    = ExcelLVLField(next(it))
    self.academic_lvl = AcademicLVLField(next(it))
  def __str__(self):
    return (
       "--- "
      f"Candidate Priority: {self.priority()}"
       " | "
      f"Pts: {self.pts()}"
       " ---\n"
      f"Name: {self.name}\n"
      f"Phone number: {self.phone}\n"
      f"Experiece: {self.years_exp} years\n"
      f"Catalan: {self.catalan_lvl}\n"
      f"Excel: {self.excel_lvl}\n"
      f"Academic level: {self.academic_lvl}\n"
    )
  def priority(self) -> str:
    if self.pts() >= 12:
      return "High Priority"
    if self.pts() >= 8:
      return "Adequate"
    return "No priority"
  def pts(self) -> float:
    return (
      1.5*self.years_exp.field
      + self.catalan_lvl.field
      + self.excel_lvl.field
      + self.academic_lvl.field
    )

- `main()`: Loads `10` `Candidate`s, and prints them by their puntuation order.
    1. Create empty `list[Candidate]` in variable `vec`.
    2. Ask user if they want to use `inputs_test`:
        - If they wrote `y`, then put an iterator over `inputs_test` in `it`.
        - If they wrote anything else, put placeholder `Noneroator` in `it`.
    3. Loop 10 times:
        1. Take the next value in `it`.
        2. Use it as paramenter to create a `Candidate` and store it in `c`.
        3. Append `c` at the end of `list` `vec`.
        4. Print `c`'s puntuation.
    4. Print visual separator.
    5. Sort `vec` via `Candidate` puntuation `pts()` in descending order.
    6. Iterate over all elements in `vec`, and print them.

In [13]:
vec = []
test = input("Use 'inputs_test'? y/n: ")
it = iter(inputs_test) if test == "y" else Nonerator()
for _ in range(0, 10):
  c = Candidate(next(it))
  vec.append(c)
  print(f"Puntuation {c.name.field}: {c.pts()}")
print(
  "\n"
  "=====================\n"
  "Sorting candidates...\n"
  "=====================\n"
)
vec.sort(key=lambda c: -c.pts())
for item in vec:
  print(item)

Use 'inputs_test'? y/n: y
Puntuation Laura Serra: 18.0
Puntuation Clara Gómez: 12.5
Puntuation Sergio López: 10.5
Input has to be either 0, 1, 2 or 3.
Input catalan lvl:
  0 → no knowledge
  1 → A1 to B1
  2 → B2
  3 → C1 or superior
0
Puntuation Candidat Error: 12.5
Puntuation Marta Ruiz: 9.0
Puntuation Marc Vidal: 18.0
Puntuation Júlia Romero: 15.5
Puntuation Pol Navarro: 13.0
Puntuation David Ferrer: 6.0
Puntuation Anna Paredes: 4.5

Sorting candidates...

--- Candidate Priority: High Priority | Pts: 18.0 ---
Name: Laura Serra
Phone number: 608112233
Experiece: 6.0 years
Catalan: C1-C2 or Native
Excel: Knows Visual Basic and Macros
Academic level: Masters or superior

--- Candidate Priority: High Priority | Pts: 18.0 ---
Name: Marc Vidal
Phone number: 612443322
Experiece: 6.0 years
Catalan: C1-C2 or Native
Excel: Knows Visual Basic and Macros
Academic level: Masters or superior

--- Candidate Priority: High Priority | Pts: 15.5 ---
Name: Júlia Romero
Phone number: 677998877
Experiec

### Exercici 2: Segmentació per risc de cistella de compra

> Treballes com a analista de dades al departament de Màrqueting Digital. L'empresa vol millorar l'efectivitat de les campanyes de recuperació de cistellas de compra abandonats i necessita identificar quins clients tenen més risc d'abandonament, quin segment de comportament presenten i quin missatge publicitari és el més adequat per a cadascun.

Disposes d'un conjunt de dades amb informació bàsica de 10 clients:

- id_client
- nom
- edat
- grup_edat
- codi_comportament

In [14]:
clients = [
  {"id_client": 1, "nom": "Laura Serra", "edat": 23, "grup_edat": "18-25", "codi_comportament": 4},
  {"id_client": 2, "nom": "Marc Vidal", "edat": 35, "grup_edat": "26-40", "codi_comportament": 3},
  {"id_client": 3, "nom": "Ana López", "edat": 29, "grup_edat": "26-40", "codi_comportament": 2},
  {"id_client": 4, "nom": "Joan Riera", "edat": 19, "grup_edat": "18-25", "codi_comportament": 2},
  {"id_client": 5, "nom": "Mónica Pérez", "edat": 47, "grup_edat": "41-60", "codi_comportament": 4},
  {"id_client": 6, "nom": "Luis García", "edat": 52, "grup_edat": "41-60", "codi_comportament": 1},
  {"id_client": 7, "nom": "Pilar Sánchez", "edat": 61, "grup_edat": "60+", "codi_comportament": 3},
  {"id_client": 8, "nom": "Eva Martín", "edat": 38, "grup_edat": "26-40", "codi_comportament": 0},
  {"id_client": 9, "nom": "Diego Romero", "edat": 24, "grup_edat": "18-25", "codi_comportament": 1},
  {"id_client": 10, "nom": "Núria Costa", "edat": 33, "grup_edat": "26-40", "codi_comportament": 4}
]

El departament de CRM ha definit manualment:

- Codis de comportament
    - 0 → Navega però no afegeix productes
    - 1 → Afegeix productes però no arriba al checkout
    - 2 → Arriba al checkout però no paga
    - 3 → Cistella de compra abandonat fa menys de 24 h
    - 4 → Cistella de compra abandonat fa més de 24 h
- Factors segons codi de comportament
    - Codi 0 → 0.10
    - Codi 1 → 0.25
    - Codi 2 → 0.50
    - Codi 3 → 0.75
    - Codi 4 → 0.90
- Factors segons grup d'edat
    - 18–25 → 1.20
    - 26–40 → 1.00
    - 41–60 → 0.85
    - 60+ → 0.70
- Fórmula del risc final
    
    `risc_final = factor codi * factor edat`
    

Un cop processats els clients, has de ser capaç de respondre:

- 1. Quins clients s'han de contactar primer en una campanya de retargeting?
- 2. Quines diferències hi ha entre clients amb el mateix comportament però d'edats diferents? (Per exemple: un client jove que abandona la cistella de compra fa 24 h té el mateix risc que un de més edat?)
- 3. Quins clients presenten tan poc interès que no val la pena invertir-hi pressupost?
- 4. Quin client seria el més rendible de recuperar?

Interpretant el risc final i el segment.

#### ⚙️ Requisits tècnics del teu programa

Hauràs de construir un sistema modular utilitzant diverses funcions i, sobretot, sense utilitzar `if/elif/else` per fer la segmentació.

**Important**

Per resoldre aquest exercici, hauràs de:

- Desenvolupar un sistema modular amb les funcions següents:
- S'ha d'implementar amb un diccionari de diccionaris, sense condicions.
    - `processar_clients` # s'ha d'alimentar de les funcions següents:
        - `obtenir_segment_i_factor`
        - `obtenir_factor_edat`
        - `calcular_risc`
    - `mostrar_dataframe` o `mostrar_diccionari`
    - `mostrar_respostes`
    - `main_marketing` # orquestra totes les funcions
- Escriure primer l'algoritme pas a pas, indicant clarament quines operacions farà cada funció.
- Escriure el codi de cada funció.

#### ✅ Resultat esperat

Un DataFrame o diccionari final amb:

- id_cliente
- nombre
- edad
- grupo_edad
- codigo_comportamiento
- segment
- missatge
- factor_codi
- factor_edat
- risc_final

I capacitat per respondre totes les preguntes de màrqueting demanades en una funció.

Defining behaviour `segment` dictionary for each behaviour code.

In [15]:
segment = {
  0: "Navigated checkout but didn't add products.",
  1: "Added products but didn't reach checkout.",
  2: "Reached checkout but didn't pay.",
  3: "Abandoned shopping cart less than 24h ago.",
  4: "Abandoned shopping cart more than 24h ago."
}

Defining `age_factor` dictionary for each age segment.

In [16]:
age_factor = {
  "18-25": 1.20,
  "26-40": 1.00,
  "41-60": 0.85,
  "60+"  : 0.70
}

Defining `behaviour_factor` dictionary for each behaviour code.

In [17]:
behaviour_factor = {
  0: 0.10,
  1: 0.25,
  2: 0.50,
  3: 0.75,
  4: 0.90
}

Defining a most insidious `message` for every combination of age segment and behaviour code.

In [18]:
message = {
  "18-25": {
    0: (
      "If they leave it at that, they didn't find what they sought."
      "Ask them to submit a descriptive wishlist."
    ),
    1: (
      "Use small discounts on some products, with a shorter countdown timer."
      "Suggest complimentary products at a timed discount."
    ),
    2: (
      "Allow them to find discount coupons about to expire."
    ),
    3: (
      "Throw more agressive discounts that will expire, remind them of their "
      "basket and offers through email or app notifications."
    ),
    4: (
      "They are likely no longer interested, remove all discounts, if they "
      "check again they will see that the timers were not a bluff, "
      "thus make the fear of losing out more substantial next time."
    )
  },
  "26-40": {
    0: (
      "If they leave it at that, use navigation data to figure "
      "out if there was something specific they sought. "
      "Ask them to submit a descriptive wishlist."
    ),
    1: (
      "Use small discounts on some products, with a countdown timer. "
      "Suggest complimentary products at a timed discount."
    ),
    2: (
      "Present discount coupons about to expire."
    ),
    3: (
      "Throw more agressive discounts that will expire, remind them "
      "of their basket and offers through email or app notifications."
    ),
    4: (
      "Remind them of the basket as each of the offers is about to expire, "
      "present worse discounts if they check those products again."
    )
  },
  "41-60": {
    0: (
      "If they leave it at that, use navigation data to "
      "figure out if there was something specific they sought."
      "Ask them to submit a descriptive wishlist."
    ),
    1: (
      "Use small discounts on some products, with a longer "
      "countdown timer, make the timer notable. "
      "Suggest complimentary products at a discount."
    ),
    2: (
      "Present discount coupons about to expire."
    ),
    3: (
      "Throw more agressive discounts that will expire or even "
      "add complimentary offers, remind them of their basket and offers "
      "through email or app notifications."
    ),
    4: (
      "Remind them of the basket as each of the offers is about to expire."
    )
  },
  "60+"  : {
    0: (
      "If they leave it at that, use navigation data throughfully to "
      "find if there was something specific they sought. "
      "Ask them to submit a descriptive wishlist."
    ),
    1: (
      "Suggest complimentary products at a discount."
    ),
    2: (
      "Present discount coupons about to expire."
    ),
    3: (
      "Throw more agressive discounts that will expire or even "
      "add complimentary offers, remind them of their basket and offers "
      "through email or app notifications."
    ),
    4: (
      "Remind them of the basket as each of the offers is about to expire. "
      "Throw more offers and complements."
    )
  }
}

`final_risk(client: dict)`: Calculates the final risk by taking the behaviour code and age group from the client dict and fetching their correspondent factors by using previously defined dictionaries.

In [19]:
def final_risk(client):
  return (
      behaviour_factor[client["codi_comportament"]]
    * age_factor[client["grup_edat"]]
  )

`get_segment(client: dict)`: Fetches from the previously defined `segment` dictionary the corresponding value to the behaviour code extracted from the `client` dictionaty.

In [20]:
def get_segment(client):
  return segment[client["codi_comportament"]]

`get_message(client: dict)`: Fetches from the previously defined `message` dictionary the corresponding value to the combination of age group and behaviour code extracted from the `client` dictionary.

In [21]:
def get_message(client):
  return message[client["grup_edat"]][client["codi_comportament"]]

`get_behaviour_factor(client: dict)`: Fetches from the previously defined `behaviour_factor` dictionary the corresponding value to the behaviour code extracted from the `client` dictionary.

In [22]:
def get_behaviour_factor(client):
  return behaviour_factor[client["codi_comportament"]]

`get_age_factor(client: dict)`: Fetches from the previously defined `age_factor` dictionary the corresponding value to the age segment extracted from the `client` dictionary.

In [23]:
def get_age_factor(client):
  return age_factor[client["grup_edat"]]

`process_clients()`: For each `client` in `clients` renames the dict fields and fetches additional data from the dictionaries using `client` entries. Appends every client to `vec` and uses the list to generate a `DataFrame`.

In [24]:
def process_clients():
  vec = []
  for client in clients:
    vec.append({
      "id_cliente"           : client["id_client"],
      "nombre"               : client["nom"],
      "edad"                 : client["edat"],
      "grupo_edad"           : client["grup_edat"],
      "codigo_comportamiento": client["codi_comportament"],
      "segment"              : get_segment(client),
      "missatge"             : get_message(client),
      "factor_codi"          : get_behaviour_factor(client),
      "factor_edat"          : get_age_factor(client),
      "risc_final"           : final_risk(client)
    })
  return pd.DataFrame(vec)

`main`: Just display the dataframe generated by `process_clients()`

In [25]:
display(process_clients())

,id_cliente,nombre,edad,grupo_edad,codigo_comportamiento,segment,missatge,factor_codi,factor_edat,risc_final
0,1,Laura Serra,23,18-25,4,Abandoned shopping cart more than 24h ago.,"They are likely no longer interested, remove a...",0.90,1.20,1.0800
1,2,Marc Vidal,35,26-40,3,Abandoned shopping cart less than 24h ago.,Throw more agressive discounts that will expir...,0.75,1.00,0.7500
2,3,Ana López,29,26-40,2,Reached checkout but didn't pay.,Present discount coupons about to expire.,0.50,1.00,0.5000
3,4,Joan Riera,19,18-25,2,Reached checkout but didn't pay.,Allow them to find discount coupons about to e...,0.50,1.20,0.6000
4,5,Mónica Pérez,47,41-60,4,Abandoned shopping cart more than 24h ago.,Remind them of the basket as each of the offer...,0.90,0.85,0.7650
5,6,Luis García,52,41-60,1,Added products but didn't reach checkout.,"Use small discounts on some products, with a l...",0.25,0.85,0.2125
6,7,Pilar Sánchez,61,60+,3,Abandoned shopping cart less than 24h ago.,Throw more agressive discounts that will expir...,0.75,0.70,0.5250
7,8,Eva Martín,38,26-40,0,Navigated checkout but didn't add products.,"If they leave it at that, use navigation data ...",0.10,1.00,0.1000
8,9,Diego Romero,24,18-25,1,Added products but didn't reach checkout.,"Use small discounts on some products, with a s...",0.25,1.20,0.3000
9,10,Núria Costa,33,26-40,4,Abandoned shopping cart more than 24h ago.,Remind them of the basket as each of the offer...,0.90,1.00,0.9000


### Exercici 1: Gestió de risc i priorització de pacients

> **Treballes com a analista de dades en un centre mèdic especialitzat**. Diversos pacients han enviat els seus símptomes a través d'un formulari online, però les dades arriben en un format **desordenat i poc estructurat**, i l'equip mèdic necessita que els organitzis per poder prioritzar l'ordre d'atenció segons la gravetat.

A continuació tens una llista amb informació real procedent del sistema:

In [26]:
pacients = [
  "Maria|45|tos, Fiebre Alta, dolor_pecho",
  "Luis|33|dolor_cabeza,faTiga",
  "Sara|67|fiebre alta, dificultad RESPIRAR, tos",
  "Jordi|52|fatiga , tos",
  "Anna|29|fiebre alta,dolor_cabeza"
]

El departament mèdic també t'ha proporcionat el següent diccionari de nivells de risc per símptoma:

In [27]:
nivells_risc = {
  "fiebre_alta": 3,
  "dificultad_respirar": 5,
  "dolor_pecho": 4,
  "tos": 1,
  "fatiga": 1,
  "dolor_cabeza": 1
}

#### 🎯 Objectius

L'objectiu és transformar aquestes dades en informació útil per als metges. A partir dels registres en brut, has de:

- Convertir cada string en una estructura neta (diccionari o similar).
    
- Separar i normalitzar la llista de símptomes:
    - minúscules
    - eliminar espais
    - dividir correctament per comes
- Assignar un risc a cada símptoma utilitzant el diccionari `nivells_risc`.
    
- Calcular el risc total de cada pacient sumant els valors.
    
- Ordenar els pacients de major a menor risc per determinar la prioritat d'atenció.
    

**⚠️ Important**

En aquest Nivell 2 **ja no et proporcionem la llista de funcions** que has de crear. Ets tu qui ha de plantejar **quines funcions necessites** i com modularitzar el procés.

Tot i així, es recomana que:

- La funció principal orquestri tots els passos.
    
- Cada subfunció faci una sola tasca concreta (neteja, conversió, càlcul, ordenació...).
- La lògica estigui separada de manera neta i coherent.
    

#### ✅ Resultat esperat

Un DataFrame o un diccionari amb:

- nom
- edat
- símptomes normalitzats
- risc_total

I un apartat final amb els pacients ordenats per prioritat (més risc → primer).

In [28]:
def process_name(s: str) -> str:
  return s.strip()

def process_age(s: str, name: str) -> int | str:
  age = s.strip()
  try: age = int(age)
  except (TypeError, ValueError):
    print(f"Failed to parse {name}'s age.")
  return age

def process_symptoms(s: str) -> list[str]:
  symptoms = s.split(',')
  result   = []
  for symptom in symptoms:
    result.append(
      re.sub(
        r"[^a-z]+",
        "_",
        symptom
          .strip()
          .lower()
      )
    )
  return result

def calculate_risk(symptoms: list[str]):
  risk_lvl = 0
  for symptom in symptoms:
    risk_lvl += nivells_risc[symptom]
  return risk_lvl

def process_patient(s: str):
  name, age, symptoms = s.split('|')
  name     = process_name(name)
  symptoms = process_symptoms(symptoms)
  return {
    "name"    : name,
    "age"     : process_age(age, name),
    "symptoms": symptoms,
    "risk_lvl": calculate_risk(symptoms)
  }

def generate_dataframe():
  vec = []
  for patient in pacients:
    vec.append(process_patient(patient))
  vec.sort(key=lambda p: -p["risk_lvl"])
  return pd.DataFrame(vec)

generate_dataframe()

,name,age,symptoms,risk_lvl
0,Sara,67,"[fiebre_alta, dificultad_respirar, tos]",9
1,Maria,45,"[tos, fiebre_alta, dolor_pecho]",8
2,Anna,29,"[fiebre_alta, dolor_cabeza]",4
3,Luis,33,"[dolor_cabeza, fatiga]",2
4,Jordi,52,"[fatiga, tos]",2


### Exercici 2: Classificació de clients de segurs

> Treballes com a analista de dades en una companyia asseguradora.
>
> L'empresa vol calcular el risc dels clients, el preu estimat de la pòlissa i la probabilitat de frau, utilitzant un codi de comportament que ha estat generat per un model intern.

Tens la següent llista de clients:

In [29]:
clients = [
  {"nom": "Ana", "edat": 42, "codi": 0},
  {"nom": "Carlos", "edat": 30, "codi": 2},
  {"nom": "Isabel", "edat": 55, "codi": 3},
  {"nom": "Jorge", "edat": 40, "codi": 1},
  {"nom": "Marta", "edat": 28, "codi": 0}
]

I aquest diccionari amb les categories del sistema:

In [30]:
nivells_seguro = {
  0: {"categoria": "Baix risc", "preu_base": 120, "fraude": 0.01},
  1: {"categoria": "Risc mitjà", "preu_base": 200, "fraude": 0.05},
  2: {"categoria": "Risc alt", "preu_base": 350, "fraude": 0.15},
  3: {"categoria": "Risc crític", "preu_base": 500, "fraude": 0.30}
}

El preu final s'ha de calcular amb la fórmula:

`preu_final = preu_base * (1 + edat/100)`

#### 🎯 Objectius

- Classificar cada client utilitzant `nivells_seguro` (`sense if/elif`).
    
- Afegir:
    - La categoria
    - El preu base
    - La probabilitat de frau
- Calcular el preu final de la pòlissa amb la fórmula donada.
    
- Crear un `DataFrame` o diccionari amb la informació final.
    
- Ordenar els clients:
    - Per probabilitat de frau (descendent).
    - Per preu final.

**Important**

Igual que a l'Exercici 1, en aquest nivell ets tu qui ha de definir les funcions necessàries. Es recomana mantenir el codi modular, on:

- Cada funció resol un pas del procés.
    
- La funció principal delega totes les tasques.

#### ✅ Resultat esperat

Un DataFrame o un diccionari amb:

- Nom
- Edat
- Categoria
- Preu_base
- Preu_final
- Probabilitat_frau
- Un apartat d'anàlisi on s'indiqui:
    - Quin client té més probabilitat de frau
    - Quin té el preu final més alt
    - Qui seria prioritari de revisar manualment

In [31]:
def process_client(c):
  lvl  = nivells_seguro[c["codi"]]
  base = lvl["preu_base"]
  age  = c["edat"]
  return {
    "Nom"              : c["nom"],
    "Edat"             : age,
    "Categoria"        : lvl["categoria"],
    "Preu_base"        : base,
    "Preu_final"       : base * (1 + age/100),
    "Probabilitat_frau": lvl["fraude"]
  }

def generate_dataframe():
  vec = []
  for client in clients:
    vec.append(process_client(client))
  return pd.DataFrame(vec)

generate_dataframe()

,Nom,Edat,Categoria,Preu_base,Preu_final,Probabilitat_frau
0,Ana,42,Baix risc,120,170.4,0.01
1,Carlos,30,Risc alt,350,455.0,0.15
2,Isabel,55,Risc crític,500,775.0,0.30
3,Jorge,40,Risc mitjà,200,280.0,0.05
4,Marta,28,Baix risc,120,153.6,0.01


- Quin client té més probabilitat de frau?

`Isabel`

- Quin té el preu final més alt?

`Isabel`

- Qui seria prioritari de revisar manualment?

`Isabel`

## Nivell 3

### Exercici 1: Generador de contrasenyes segures

> Ets analista de dades dins el departament de **Ciberseguretat** d'una empresa tecnològica. S'ha detectat que la majoria de treballadors continuen utilitzant contrasenyes molt dèbils com “Asdf1234”, dates d'aniversari o combinacions fàcils d'endevinar. Per aquest motiu, el responsable del departament t'ha demanat desenvolupar un **generador de contrasenyes segures**, totalment parametritzable i modular.

#### 📝 Requisits del generador

Has de crear una funció principal i totes les subfuncions necessàries per:

- Generar contrasenyes de longitud variable.
    
- Decidir si han d'incloure:
    - Majúscules
    - Minúscules
    - Números
    - Signes especials
- Assegurar-te que **com a mínim apareix un caràcter de cada tipus seleccionat**.
    
- Aleatoritzar completament la contrasenya.
    
- Retornar la contrasenya generada.
    

#### 📍 Nota important

Explora el funcionament del mòdul `random` de la llibreria [numpy](https://www.cybrosys.com/blog/random-module-in-numpy)

La funció principal del generador ha de tenir valors per defecte, exactament els següents:

- `majuscules=True`
- `minuscules=True`
- `numeros=True`
- `signes=False`

Aquests paràmetres són obligatoris i formen part del disseny del sistema.

In [32]:
def ascii_range(b: str, e: str) -> str:
  return "".join(map(chr, range(
      ord(b),
      ord(e) + 1
  )))

class PwdGen:
  NMS = [
    "upper case",
    "lower case",
    "digit",
    "symbol"
  ]
  STR = [
    ascii_range('A', 'Z'),
    ascii_range('a', 'z'),
    ascii_range('0', '9'),
    ascii_range('!', '/')
  + ascii_range(':', '@')
  + ascii_range('[', '`')
  + ascii_range('{', '~')
  ]
  REG = [
    re.compile(r"[A-Z]"),
    re.compile(r"[a-z]"),
    re.compile(r"[0-9]"),
    re.compile(r"[!-/:-@[-`{-~]")
  ]
  @staticmethod
  def generate(
    length=12,
    upper=True,
    lower=True,
    digit=True,
    symbl=False
  ) -> str:
    enabled = [
      i for i, v in
      enumerate([upper, lower, digit, symbl])
      if v
    ]
    if not enabled: raise Exception(
      "error: no character set enabled."
    )
    if len(enabled) > length: raise Exception(
      f"error: password length too short. "
      f"{len(enabled)} characters required for "
      f"{", ".join(PwdGen.NMS[i] for i in enabled)}. "
      f"Requested length: {length}"
    )
    pool = "".join(PwdGen.STR[i] for i in enabled)
    chck = [PwdGen.REG[i] for i in enabled]
    while True:
      pwd = "".join(
        pool[r] for r in
        np.random.randint(0, len(pool), length)
      )
      if all(p.search(pwd) for p in chck):
        return pwd

In [33]:
print(
  PwdGen.generate(
    12,
    upper=True,
    lower=True,
    digit=True,
    symbl=True
  )
)

@_Sv\'*6l-&=


### Exercici 2: Processament automàtic de dades esportives

> Ara formes part d'un equip d'analítica dins una organització esportiva que gestiona dades de la lliga catalana. La teva companya de feina vol automatitzar la lectura del seu arxiu històric de resultats i extreure les estadístiques més importants. T'ha demanat fer un petit sistema en Python que processi el fitxer 📄 `històric partits.txt`

#### 🎯 Objectius

Mitjançant un codi **modular, clar i documentat**, el teu programa ha de calcular:

1. Quin equip és el més golejador.    
2. Quin és el més golejat.
3. El nombre total de gols que ha fet cada equip.
4. La classificació final ordenada per punts on cada victòria suma 3 `pts`, cada empat 1 `pts` i cada derrota 0 `pts`.

En aquest nivell no es dona cap funció. Ets tu qui ha de dissenyar un flux complet. El codi ha de reflectir una bona organització interna.

In [34]:
uploaded = files.upload()

Saving historic partits.txt to historic partits.txt


In [35]:
text = uploaded["historic partits.txt"].decode("utf-8")

In [36]:
class Games:
  def __init__(self, games: str):
    """
Load and process the games database into a `DataFrame`.
#### Args:
- `games: str`: text from `historic partits.txt` \
or of similar format.
    """
    games = Games._games(games)
    teams = Games._teams(games)
    data  = {
      team: {
        "team_goals": 0,
        "enemy_goals": 0,
        "pts": 0
      } for team in teams
    }
    self.data = pd.DataFrame(
      data=games,
      columns=["team0","team1","score0","score1"]
    )
    for team0, team1, score0, score1 in games:
      data[team0]["team_goals"] += score0
      data[team0]["enemy_goals"] += score1
      data[team1]["team_goals"] += score1
      data[team1]["enemy_goals"] += score0
      if score0 == score1:
        data[team0]["pts"] += 1
        data[team1]["pts"] += 1
      elif score0 > score1:
        data[team0]["pts"] += 3
      else:
        data[team1]["pts"] += 3
    self.summary = (pd
      .DataFrame(data).T # transpose
      .sort_values(by=["pts"], ascending= False)
    )
    print(
      f"Team with most goals is: "
      f"{self.summary.team_goals.idxmax()}"
    )
    print(
      f"Team with most enemy goals is: "
      f"{self.summary.enemy_goals.idxmax()}"
    )
  @staticmethod
  def _teams(games: list[tuple[str, str, int, int]]) -> Iterator[str]:
    """
Iterator over `teams` in `games`.

#### Args:
- `games`: list of `team0, team1, _, _: tuple[str, str, _, _]`
#### Returns:
- `Iterator[str]`: a sequence of teams.
    """
    for team0, team1, _, _ in games:
      yield team0
      yield team1
  @staticmethod
  def _games(text: str) -> list[tuple[str, str, int, int]]:
    """
Parses the text into a `list` of `games`.

#### Args:
- `text`: raw text from `historic partits.txt` or compatible format.
#### Returns:
- `team0, team1, score0, score1: list[tuple[str, str, int, int]]`: \
a sequence of games.
    """
    return [
      Games._process(line)
      for line in text.split("\n")
      if line != ""
    ]
  @staticmethod
  def _process(game: str):
    """
Parses a line of text describing a game into its teams and scores.

#### Args:
- `game`: a single line of text containing game data.
#### Returns:
- `team0, team1, score0, socre1: tuple[str, str, int, int]`: \
extracted game data.
    """
    team0, score, team1 = map(
      str.strip,
      re.split(r"(\d+-\d+)", game)
    )
    score0, score1 = map(
      int,
      score.split("-")
    )
    return (
      team0,
      team1,
      score0,
      score1
    )

In [37]:
g = Games(text)
display(g.summary)
display(g.data)

Team with most goals is: Figueres
Team with most enemy goals is: Vilafranca


,team_goals,enemy_goals,pts
Girona FC,139,94,96
Llagostera,159,142,94
Sabadell,141,121,85
Cornellà,147,146,82
RCD Espanyol,131,144,80
Figueres,161,154,79
Lleida Esportiu,129,133,75
Terrassa,147,164,74
FC Barcelona,125,115,73
Vilafranca,157,172,72


,team0,team1,score0,score1
0,Manlleu,Granollers,0,1
1,Vilafranca,Terrassa,4,5
2,Olot,Granollers,5,0
3,Vilafranca,Manlleu,5,5
4,Cornellà,Prat,5,1
...,...,...,...,...
495,Nàstic de Tarragona,Olot,1,3
496,Badalona,Prat,5,4
497,Olot,Llagostera,4,5
498,FC Barcelona,Cornellà,3,1
